In [1]:
import geopandas as gpd
from shapely.geometry import shape
import shapely
import pandas as pd
import folium
import osmnx as ox
from shapely.ops import substring
import networkx as nx
from shapely.ops import nearest_points
from shapely.geometry import Point, LineString, MultiLineString
from folium import plugins

In [2]:
ta = gpd.read_file('data/chc-boundaries/territorial-authority-2021-generalised.gpkg', engine='pyogrio')
sa2 = gpd.read_file('data/chc-boundaries/sa2/statistical-area-2-2023-generalised.shp')
canopy = gpd.read_file('data/canopy.gdb')
property_full = gpd.read_file('output/hedonic_gdf.gpkg')

pd.set_option('display.max_columns', None)

ta_chc = ta[ta['TA2021_V1_00_NAME_ASCII'] == 'Christchurch City']
sa2_chc = gpd.clip(sa2, ta_chc)
boundary = sa2_chc[sa2_chc['SA22023__2'].str.lower().str.contains('ilam')]

property = gpd.clip(property_full, boundary)
TARGET_CRS = 2193

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/pyogrio/raw.py:198: RuntimeWarning: organizePolygons() received a polygon with more than 100 parts. The processing may be really slow.  You can skip the processing by setting METHOD=SKIP, or only make it analyze counter-clock wise parts by setting METHOD=ONLY_CCW if you can assume that the outline of holes is counter-clock wise defined
  return ogr_read(


In [3]:
G = ox.graph_from_place(
  'Christchurch, New Zealand',
  network_type='drive',
  simplify=True,
)

G = ox.project_graph(G, to_crs=TARGET_CRS)

edges = ox.graph_to_gdfs(G, nodes=False)
edges = edges.to_crs(property.crs)

def nearest_street_and_point(point, edges_gdf):
    # find index of nearest street segment
    idx = edges_gdf.geometry.distance(point).idxmin()
    street_geom = edges_gdf.loc[idx].geometry

    # project property onto street
    proj_dist = street_geom.project(point)
    access_point = street_geom.interpolate(proj_dist)

    return street_geom, access_point
  
property[["front_street_geom", "access_point"]] = (
    property.geometry
    .apply(lambda p: nearest_street_and_point(p, edges))
    .apply(pd.Series)
)

/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)
/Users/rujalshrestha/Projects/chc-property-prices/venv/lib/python3.13/site-packages/geopandas/geodataframe.py:1968: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  super().__setitem__(key, value)


In [4]:
# property.to_file('output/property_with_access_points.gpkg', driver='GPKG')

# claude

In [129]:

def isodistance_from_access_point(property_row, property_crs, G, max_dist=200, verbose=False):

    access_pt = property_row['access_point']
    front_street = property_row['front_street_geom']

    if verbose:
        print("\n==============================")
        print("Starting isodistance computation")
        print("==============================")

    # --------------------------------------------------
    # Project graph and inputs
    # --------------------------------------------------
    G_proj = ox.project_graph(G)

    gdf_temp = gpd.GeoDataFrame(
        {'geometry': [access_pt, front_street]},
        crs=property_crs
    ).to_crs(G_proj.graph['crs'])

    access_pt_proj = gdf_temp.iloc[0].geometry
    front_street_proj = gdf_temp.iloc[1].geometry

    if verbose:
        print("Graph CRS:", G_proj.graph['crs'])
        print("Access point projected:", access_pt_proj)

    # --------------------------------------------------
    # Find matching edge
    # --------------------------------------------------
    best_match = None
    min_distance = float('inf')

    for u, v, k, data in G_proj.edges(keys=True, data=True):
        if 'geometry' not in data:
            continue
        d = data['geometry'].distance(front_street_proj)
        if d < min_distance:
            min_distance = d
            best_match = (u, v, k, data)

    if verbose:
        print(f"Closest edge distance: {min_distance:.2f} m")

    if best_match is None or min_distance > 50:
        if verbose:
            print("❌ No suitable matching edge found")
        return MultiLineString([])

    u, v, k, data = best_match
    edge_geom = data['geometry']
    edge_length = data['length']

    if verbose:
        print(f"Matched edge: {u} ↔ {v}")
        print(f"Edge length: {edge_length:.2f} m")

    # --------------------------------------------------
    # Project access point onto edge
    # --------------------------------------------------
    proj_dist = edge_geom.project(access_pt_proj)
    proj_dist = min(max(proj_dist, 0), edge_length)

    dist_to_u = proj_dist
    dist_to_v = edge_length - proj_dist

    if verbose:
        print(f"Projection distance on edge: {proj_dist:.2f} m")
        print(f"Distance to u: {dist_to_u:.2f} m")
        print(f"Distance to v: {dist_to_v:.2f} m")

    # --------------------------------------------------
    # Build working graph (UNDIRECTED COPY)
    # --------------------------------------------------
    G_work = G_proj.to_undirected().copy()

    # --------------------------------------------------
    # Insert access node if needed
    # --------------------------------------------------
    if dist_to_u < 0.5:
        start_node = u
        if verbose:
            print("Access point coincides with node u")

    elif dist_to_v < 0.5:
        start_node = v
        if verbose:
            print("Access point coincides with node v")

    else:
        access_node = 'access_point'
        G_work.add_node(access_node, x=access_pt_proj.x, y=access_pt_proj.y)

        geom_to_u = substring(edge_geom, 0, proj_dist)
        geom_to_v = substring(edge_geom, proj_dist, edge_length)

        G_work.add_edge(access_node, u, length=dist_to_u, geometry=geom_to_u)
        G_work.add_edge(access_node, v, length=dist_to_v, geometry=geom_to_v)

        if G_work.has_edge(u, v):
            G_work.remove_edge(u, v)

        start_node = access_node

        if verbose:
            print("Inserted synthetic access node")

    # --------------------------------------------------
    # Dijkstra (single source, cumulative distance)
    # --------------------------------------------------
    lengths = nx.single_source_dijkstra_path_length(
        G_work,
        start_node,
        cutoff=max_dist,
        weight='length'
    )

    if verbose:
        print(f"Reachable nodes: {len(lengths)}")
        print("Sample distances:", list(lengths.items())[:10])

    # --------------------------------------------------
    # Restrict to connected reachable subgraph
    # --------------------------------------------------
    reachable_nodes = set(lengths.keys())
    G_reachable = G_work.subgraph(reachable_nodes)

    if verbose:
        print(f"Reachable edges: {G_reachable.number_of_edges()}")

    # --------------------------------------------------
    # Collect geometries (NO LEAKAGE)
    # --------------------------------------------------
    lines = []
    edges_checked = 0
    edges_added = 0

    for u_node, v_node, data in G_reachable.edges(data=True):
        edges_checked += 1

        geom = data.get('geometry')
        if geom is None:
            u_pt = Point(G_reachable.nodes[u_node]['x'], G_reachable.nodes[u_node]['y'])
            v_pt = Point(G_reachable.nodes[v_node]['x'], G_reachable.nodes[v_node]['y'])
            geom = LineString([u_pt, v_pt])

        du = lengths[u_node]
        dv = lengths[v_node]
        edge_len = data.get('length', geom.length)

        if du <= max_dist and dv <= max_dist:
            lines.append(geom)
            edges_added += 1

        elif du <= max_dist < dv:
            remaining = max_dist - du
            if remaining > 0:
                lines.append(substring(geom, 0, min(remaining, edge_len)))
                edges_added += 1

        elif dv <= max_dist < du:
            remaining = max_dist - dv
            start = max(edge_len - remaining, 0)
            lines.append(substring(geom, start, edge_len))
            edges_added += 1

    if verbose:
        print(f"Edges checked: {edges_checked}")
        print(f"Edges added: {edges_added}")
        print(f"Collected line segments: {len(lines)}")
        if lines:
            print(f"Total collected length: {sum(l.length for l in lines):.2f} m")

    if not lines:
        return MultiLineString([])

    result = MultiLineString(lines)
    return gpd.GeoSeries([result], crs=G_proj.graph['crs']).to_crs(property_crs).iloc[0]


def process_all_properties(properties_gdf, G, max_dist=200):

    results = []

    for idx, row in properties_gdf.iterrows():
        if idx % 100 == 0:
            print(f"Processing property {idx}/{len(properties_gdf)}")

        reach = isodistance_from_access_point(
            row,
            properties_gdf.crs,
            G,
            max_dist,
            verbose=False
        )
        results.append(reach)

    out = properties_gdf.copy()
    out['street_reach'] = results
    return out


# modified code

In [101]:
def isodistance_from_access_point(property_row, property_crs, G, max_dist=200, verbose=False):

    access_pt = property_row['access_point']
    front_street = property_row['front_street_geom']

    # --------------------------------------------------
    # Project graph and inputs
    # --------------------------------------------------
    G_proj = ox.project_graph(G)

    gdf_temp = gpd.GeoDataFrame(
        {'geometry': [access_pt, front_street]},
        crs=property_crs
    ).to_crs(G_proj.graph['crs'])

    access_pt_proj = gdf_temp.iloc[0].geometry
    front_street_proj = gdf_temp.iloc[1].geometry

    # --------------------------------------------------
    # Find matching edge
    # --------------------------------------------------
    best_match = None
    min_distance = float('inf')

    for u, v, k, data in G_proj.edges(keys=True, data=True):
        if 'geometry' not in data:
            continue
        d = data['geometry'].distance(front_street_proj)
        if d < min_distance:
            min_distance = d
            best_match = (u, v, k, data)

    if best_match is None or min_distance > 50:
        return MultiLineString([])

    u, v, k, data = best_match
    edge_geom = data['geometry']
    edge_length = data['length']

    # --------------------------------------------------
    # Project access point onto edge
    # --------------------------------------------------
    proj_dist = edge_geom.project(access_pt_proj)
    
    proj_dist = min(max(proj_dist, 0), edge_length)

    dist_to_u = proj_dist
    dist_to_v = edge_length - proj_dist
    
    # --------------------------------------------------
    # Build working graph (UNDIRECTED COPY)
    # --------------------------------------------------
    G_work = G_proj.to_undirected().copy()

    # --------------------------------------------------
    # Insert access node if needed
    # --------------------------------------------------
    if dist_to_u < 0.5:
        start_node = u
        
    elif dist_to_v < 0.5:
        start_node = v
        
    else:
        access_node = 'access_point'
        G_work.add_node(access_node, x=access_pt_proj.x, y=access_pt_proj.y)

        geom_to_u = substring(edge_geom, 0, proj_dist)
        geom_to_v = substring(edge_geom, proj_dist, edge_length)

        G_work.add_edge(access_node, u, length=dist_to_u, geometry=geom_to_u)
        G_work.add_edge(access_node, v, length=dist_to_v, geometry=geom_to_v)

        if G_work.has_edge(u, v):
            G_work.remove_edge(u, v)
            
        start_node = access_node

    # --------------------------------------------------
    # Dijkstra (single source, cumulative distance)
    # --------------------------------------------------
    lengths = nx.single_source_dijkstra_path_length(
        G_work,
        start_node,
        cutoff=max_dist,
        weight='length'
    )

    reachable_nodes = set(lengths.keys())

    # --------------------------------------------------
    # FIX 1: restrict to connected reachable subgraph
    # --------------------------------------------------
    G_reachable = G_work.subgraph(reachable_nodes)

    # --------------------------------------------------
    # Collect geometries (NO LEAKAGE)
    # --------------------------------------------------
    lines = []

    for u_node, v_node, data in G_reachable.edges(data=True):
                
        geom = data.get('geometry')
        if geom is None:
            u_pt = Point(G_reachable.nodes[u_node]['x'], G_reachable.nodes[u_node]['y'])
            v_pt = Point(G_reachable.nodes[v_node]['x'], G_reachable.nodes[v_node]['y'])
            geom = LineString([u_pt, v_pt])

        du = lengths[u_node]
        dv = lengths[v_node]
        edge_len = data.get('length', geom.length)

        if du <= max_dist and dv <= max_dist:
            lines.append(geom)

        elif du <= max_dist < dv:
            remaining = max_dist - du
            if remaining > 0:
                lines.append(substring(geom, 0, min(remaining, edge_len)))

        elif dv <= max_dist < du:
            remaining = max_dist - dv
            start = max(edge_len - remaining, 0)
            lines.append(substring(geom, start, edge_len))

    if not lines:
        return MultiLineString([])


    result = MultiLineString(lines)
    return gpd.GeoSeries([result], crs=G_proj.graph['crs']).to_crs(property_crs).iloc[0]

# modified 2

In [209]:
def isodistance_from_access_point(property_row, property_crs, G, max_dist=200):

    access_pt = property_row['access_point']
    front_street = property_row['front_street_geom']
    
    G_proj = ox.project_graph(G, to_crs="EPSG:2193")

    gdf_temp = gpd.GeoDataFrame(
        {'geometry': [access_pt, front_street]},
        crs=property_crs
    ).to_crs("EPSG:2193")

    access_pt_proj = gdf_temp.iloc[0].geometry
    front_street_proj = gdf_temp.iloc[1].geometry

    # Find matching edge
    best_match = None
    min_distance = float('inf')

    for u, v, k, data in G_proj.edges(keys=True, data=True):
        if 'geometry' not in data:
            continue
        d = data['geometry'].distance(front_street_proj)
        if d < min_distance:
            min_distance = d
            best_match = (u, v, k, data)

    if best_match is None or min_distance > 50:
        return MultiLineString([])

    u, v, k, data = best_match
    edge_geom = data['geometry']
    edge_length = data['length']
    
    proj_dist = edge_geom.project(access_pt_proj)
    proj_dist = min(max(proj_dist, 0), edge_length)

    dist_to_u = proj_dist
    dist_to_v = edge_length - proj_dist
    
    print("\n" + "="*70)
    print("DEBUGGING PROPERTY")
    print("="*70)
    print(f"Matched edge: u={u}, v={v}")
    print(f"Edge length: {edge_length:.2f}m")
    print(f"Distance to u: {dist_to_u:.2f}m")
    print(f"Distance to v: {dist_to_v:.2f}m")
    
    G_work = G_proj.to_undirected().copy()

    if dist_to_u < 0.5:
        start_node = u
        print(f"Using u as start node")
        
    elif dist_to_v < 0.5:
        start_node = v
        print(f"Using v as start node")
        
    else:
        access_node = 'access_point'
        G_work.add_node(access_node, x=access_pt_proj.x, y=access_pt_proj.y)

        geom_to_u = substring(edge_geom, 0, proj_dist)
        geom_to_v = substring(edge_geom, proj_dist, edge_length)

        G_work.add_edge(access_node, u, length=dist_to_u, geometry=geom_to_u)
        G_work.add_edge(access_node, v, length=dist_to_v, geometry=geom_to_v)

        if G_work.has_edge(u, v):
            G_work.remove_edge(u, v)
            
        start_node = access_node
        print(f"Created access_point node")
        print(f"  Connected to u with {dist_to_u:.2f}m")
        print(f"  Connected to v with {dist_to_v:.2f}m")

    # Check neighbors before Dijkstra
    neighbors = list(G_work.neighbors(start_node))
    print(f"\nStart node has {len(neighbors)} neighbors: {neighbors}")
    
    lengths = nx.single_source_dijkstra_path_length(
        G_work,
        start_node,
        cutoff=max_dist,
        weight='length'
    )

    print(f"\nDijkstra reached {len(lengths)} nodes")
    print(f"Distance range: 0.0 to {max(lengths.values()):.2f}m")
    
    # Check if u and v were reached
    print(f"\nNode u distance: {lengths.get(u, 'NOT REACHED')}")
    print(f"Node v distance: {lengths.get(v, 'NOT REACHED')}")
    
    # Check neighbors of u and v
    u_neighbors = list(G_work.neighbors(u))
    v_neighbors = list(G_work.neighbors(v))
    
    print(f"\nNode u has {len(u_neighbors)} neighbors")
    u_reached = sum(1 for n in u_neighbors if n in lengths)
    print(f"  {u_reached} of them are reachable")
    
    print(f"\nNode v has {len(v_neighbors)} neighbors")
    v_reached = sum(1 for n in v_neighbors if n in lengths)
    print(f"  {v_reached} of them are reachable")
    
    # Show which neighbors were NOT reached
    print(f"\nUnreached neighbors of u: {[n for n in u_neighbors if n not in lengths]}")
    print(f"Unreached neighbors of v: {[n for n in v_neighbors if n not in lengths]}")
    
    # Check edges from u and v
    print(f"\nEdges from u:")
    for neighbor in u_neighbors:
        edge_data = G_work.get_edge_data(u, neighbor)
        if edge_data:
            # Handle MultiGraph (dict of dicts) vs Graph (single dict)
            if isinstance(edge_data, dict) and 'length' in edge_data:
                length = edge_data['length']
            else:
                length = list(edge_data.values())[0].get('length', 'NO LENGTH')
            reached = "✓" if neighbor in lengths else "✗"
            print(f"  u → {neighbor}: {length:.2f}m {reached}")
    
    print(f"\nEdges from v:")
    for neighbor in v_neighbors:
        edge_data = G_work.get_edge_data(v, neighbor)
        if edge_data:
            if isinstance(edge_data, dict) and 'length' in edge_data:
                length = edge_data['length']
            else:
                length = list(edge_data.values())[0].get('length', 'NO LENGTH')
            reached = "✓" if neighbor in lengths else "✗"
            print(f"  v → {neighbor}: {length:.2f}m {reached}")
    
    # Continue with normal collection...
    lines = []
    edges_added = {'full': 0, 'clipped_from_u': 0, 'clipped_from_v': 0, 'skipped': 0}

    for u_node, v_node, data in G_work.edges(data=True):
        
        du = lengths.get(u_node, float('inf'))
        dv = lengths.get(v_node, float('inf'))
        
        if du > max_dist and dv > max_dist:
            edges_added['skipped'] += 1
            continue
                
        geom = data.get('geometry')
        if geom is None:
            u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
            v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
            geom = LineString([u_pt, v_pt])

        edge_len = data.get('length', geom.length)

        if du <= max_dist and dv <= max_dist:
            lines.append(geom)
            edges_added['full'] += 1

        elif du <= max_dist < dv:
            remaining = max_dist - du
            if remaining > 0:
                u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
                v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
                geom_start = Point(geom.coords[0])
                
                if geom_start.distance(u_pt) < geom_start.distance(v_pt):
                    clipped = substring(geom, 0, min(remaining, edge_len))
                else:
                    start_dist = max(0, edge_len - remaining)
                    clipped = substring(geom, start_dist, edge_len)
                
                if clipped and not clipped.is_empty:
                    lines.append(clipped)
                    edges_added['clipped_from_u'] += 1

        elif dv <= max_dist < du:
            remaining = max_dist - dv
            if remaining > 0:
                u_pt = Point(G_work.nodes[u_node]['x'], G_work.nodes[u_node]['y'])
                v_pt = Point(G_work.nodes[v_node]['x'], G_work.nodes[v_node]['y'])
                
                geom_start = Point(geom.coords[0])
                
                if geom_start.distance(v_pt) < geom_start.distance(u_pt):
                    clipped = substring(geom, 0, min(remaining, edge_len))
                else:
                    start_dist = max(0, edge_len - remaining)
                    clipped = substring(geom, start_dist, edge_len)
                
                if clipped and not clipped.is_empty:
                    lines.append(clipped)
                    edges_added['clipped_from_v'] += 1

    print(f"\n📊 EDGE SUMMARY:")
    print(f"  Full edges: {edges_added['full']}")
    print(f"  Clipped from u: {edges_added['clipped_from_u']}")
    print(f"  Clipped from v: {edges_added['clipped_from_v']}")
    print(f"  Skipped (both unreachable): {edges_added['skipped']}")
    print(f"  Total collected: {len(lines)}")

    if not lines:
        return MultiLineString([])

    result = MultiLineString(lines)
    return gpd.GeoSeries([result], crs=G_proj.graph['crs']).to_crs(property_crs).iloc[0]

# test single

In [213]:
row = property.iloc[122]   # change index if needed

In [214]:
# -----------------------------
# COMPUTE REACH
# -----------------------------
reach = isodistance_from_access_point(
    property_row=row,
    property_crs=property.crs,
    G=G,
    max_dist=200
)

# -----------------------------
# REPROJECT GEOMETRIES FOR FOLIUM
# -----------------------------
prop = gpd.GeoSeries([row.geometry], crs=property.crs).to_crs(4326).iloc[0]
access = gpd.GeoSeries([row['access_point']], crs=property.crs).to_crs(4326).iloc[0]
front = gpd.GeoSeries([row['front_street_geom']], crs=property.crs).to_crs(4326).iloc[0]
reach_wgs = gpd.GeoSeries([reach], crs=property.crs).to_crs(4326).iloc[0]

# -----------------------------
# CREATE MAP
# -----------------------------
m = folium.Map(
    location=[prop.y, prop.x],
    zoom_start=16,
    tiles="CartoDB positron"
)

# -----------------------------
# PROPERTY (GREEN)
# -----------------------------
folium.CircleMarker(
    [prop.y, prop.x],
    radius=6,
    color="green",
    fill=True,
    fill_opacity=0.8,
    popup="Property"
).add_to(m)

# -----------------------------
# ACCESS POINT (BLUE)
# -----------------------------
folium.CircleMarker(
    [access.y, access.x],
    radius=4,
    color="blue",
    fill=True,
    fill_opacity=0.9,
    popup="Access Point"
).add_to(m)

# -----------------------------
# PROPERTY → ACCESS (DASHED)
# -----------------------------
if prop.distance(access) > 0.1:
    folium.PolyLine(
        [(prop.y, prop.x), (access.y, access.x)],
        color="purple",
        weight=2,
        opacity=0.6,
        dash_array="5, 5",
        popup="Property to Access"
    ).add_to(m)

# -----------------------------
# FRONT STREET (OPTIONAL)
# -----------------------------
folium.PolyLine(
    [(y, x) for x, y in front.coords],
    color="purple",
    weight=15,
    opacity=0.3,
    popup="Front Street"
).add_to(m)

# -----------------------------
# REACH (RED)
# -----------------------------
if not reach_wgs.is_empty:
    folium.GeoJson(
        gpd.GeoSeries([reach_wgs]).to_json(),
        style_function=lambda x: {
            "color": "red",
            "weight": 3,
            "opacity": 0.6
        },
        popup="Reachable Streets"
    ).add_to(m)

# -----------------------------
# DISPLAY
# -----------------------------
m



DEBUGGING PROPERTY
Matched edge: u=6874892393, v=124257561
Edge length: 103.94m
Distance to u: 103.94m
Distance to v: 0.00m
Using v as start node

Start node has 3 neighbors: [6874892393, 124256350, 124257199]

Dijkstra reached 5 nodes
Distance range: 0.0 to 183.66m

Node u distance: 103.93884911074025
Node v distance: 0

Node u has 3 neighbors
  2 of them are reachable

Node v has 3 neighbors
  3 of them are reachable

Unreached neighbors of u: [416309518]
Unreached neighbors of v: []

Edges from u:
  u → 416309518: 216.72m ✗
  u → 6874892394: 59.35m ✓
  u → 124257561: 103.94m ✓

Edges from v:
  v → 6874892393: 103.94m ✓
  v → 124256350: 183.66m ✓
  v → 124257199: 65.55m ✓

📊 EDGE SUMMARY:
  Full edges: 4
  Clipped from u: 2
  Clipped from v: 5
  Skipped (both unreachable): 14561
  Total collected: 11


# test multiple

In [ ]:
# pick 10 random rows from your property GeoDataFrame
sample_props = property.sample(10, random_state=42)

sample_results = []

for idx, row in sample_props.iterrows():
    reach = isodistance_from_access_point(
        property_row=row,
        property_crs=property.crs,
        G=G,
        max_dist=250,
        verbose=False
    )
    sample_results.append((row, reach))

for row, reach in sample_results:

    # reproject geometries for folium
    prop = gpd.GeoSeries([row.geometry], crs=property.crs).to_crs(4326).iloc[0]
    access = gpd.GeoSeries([row['access_point']], crs=property.crs).to_crs(4326).iloc[0]
    front = gpd.GeoSeries([row['front_street_geom']], crs=property.crs).to_crs(4326).iloc[0]
    reach_wgs = gpd.GeoSeries([reach], crs=property.crs).to_crs(4326).iloc[0]

    # property (green)
    folium.CircleMarker(
        [prop.y, prop.x],
        radius=6,
        color="green",
        fill=True,
        fill_opacity=0.8,
        popup="Property"
    ).add_to(m)

    # access point (blue)
    folium.CircleMarker(
        [access.y, access.x],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.9,
        popup="Access Point"
    ).add_to(m)

    # Connection from property to access point (dashed line)
    if prop.distance(access) > 0.1:  # If they're not the same point
        folium.PolyLine(
            [(prop.y, prop.x), (access.y, access.x)],
            color="purple",
            weight=2,
            opacity=0.6,
            dash_array="5, 5",
            popup="Property to Access"
        ).add_to(m)

    # front street (orange) - uncomment to see
    # folium.PolyLine(
    #     [(y, x) for x, y in front.coords],
    #     color="orange",
    #     weight=4,
    #     opacity=0.5,
    #     popup="Front Street"
    # ).add_to(m)

    # reach (red)
    if not reach_wgs.is_empty:
        folium.GeoJson(
            gpd.GeoSeries([reach_wgs]).to_json(),
            style_function=lambda x: {
                "color": "red",
                "weight": 3,
                "opacity": 0.6
            },
            popup="Reachable Streets"
        ).add_to(m)

m

In [118]:
# ---------------------------------------------
# Compute isodistance for ALL properties
# ---------------------------------------------

def compute_isodistance_all_properties(
    properties_gdf,
    G,
    max_dist=200,
    verbose_every=100
):
    results = []

    for idx, row in properties_gdf.iterrows():

        if idx % verbose_every == 0:
            print(f"Processing property {idx + 1} / {len(properties_gdf)}")

        reach = isodistance_from_access_point(
            property_row=row,
            property_crs=properties_gdf.crs,
            G=G,
            max_dist=max_dist,
            verbose=False
        )

        results.append(reach)

    out = properties_gdf.copy()
    out["street_reach"] = results

    return out


# run on all property

In [119]:
property_with_isodistance = compute_isodistance_all_properties(
    properties_gdf=property,
    G=G,
    max_dist=200,
    verbose_every=100
)

Processing property 3001 / 247
Processing property 3201 / 247


In [229]:
import random
import geopandas as gpd
import folium

# ---------------------------------------------
# Pick 10 random properties
# ---------------------------------------------
sample_rows = property_with_isodistance.sample(5, random_state=None)

# ---------------------------------------------
# Reproject once for Folium
# ---------------------------------------------
prop_gdf = gpd.GeoDataFrame(
    geometry=sample_rows.geometry,
    crs=property_with_isodistance.crs
).to_crs(4326)

access_gdf = gpd.GeoDataFrame(
    geometry=sample_rows["access_point"],
    crs=property_with_isodistance.crs
).to_crs(4326)

reach_gdf = gpd.GeoDataFrame(
    geometry=sample_rows["street_reach"],
    crs=property_with_isodistance.crs
).to_crs(4326)

front_gdf = gpd.GeoDataFrame(
    geometry=sample_rows["front_street_geom"],
    crs=property_with_isodistance.crs
).to_crs(4326)

# ---------------------------------------------
# Initialise map (centered on mean location)
# ---------------------------------------------
center = prop_gdf.union_all().centroid

m = folium.Map(
    location=[center.y, center.x],
    zoom_start=15,
    tiles="CartoDB positron"
)

# ---------------------------------------------
# Plot street reach (RED)
# ---------------------------------------------
folium.GeoJson(
    reach_gdf.to_json(),
    name="Street Reach (Isodistance)",
    style_function=lambda x: {
        "color": "red",
        "weight": 3,
        "opacity": 0.7
    }
).add_to(m)

# ---------------------------------------------
# Plot frontage streets (ORANGE)
# ---------------------------------------------
folium.GeoJson(
    front_gdf.to_json(),
    name="Frontage Streets",
    style_function=lambda x: {
        "color": "orange",
        "weight": 4,
        "opacity": 0.8
    }
).add_to(m)

# ---------------------------------------------
# Plot properties (GREEN)
# ---------------------------------------------
for geom in prop_gdf.geometry:
    folium.CircleMarker(
        location=[geom.y, geom.x],
        radius=6,
        color="darkgreen",
        fill=True,
        fill_color="lightgreen",
        fill_opacity=0.9
    ).add_to(m)

# ---------------------------------------------
# Plot access points (BLUE)
# ---------------------------------------------
for geom in access_gdf.geometry:
    folium.CircleMarker(
        location=[geom.y, geom.x],
        radius=4,
        color="blue",
        fill=True,
        fill_opacity=0.9
    ).add_to(m)

# ---------------------------------------------
# Controls
# ---------------------------------------------
folium.LayerControl(collapsed=False).add_to(m)

m
